# Project 08: Route Requests with a LangGraph Agent Graph

Build a small LangGraph graph that routes a customer request to a billing or technical specialist, then checks the answer with a guardrail that blocks large refunds.

Run top to bottom. Each step matches the README.

In [ ]:
import os, subprocess
REPO = "TharunPatel20/GenAI-Engineer"   # replace with your GitHub repo, e.g. "your-name/ai-engineer-course"
PROJECT = "projects/08-langgraph-agents-mcp"
if REPO.startswith("<"):
    print("Repo not set. Upload the project files with the Files panel (left sidebar), into a folder named 08-langgraph-agents-mcp, then run os.chdir('08-langgraph-agents-mcp') before Step 1.")
else:
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"], check=True)
    os.chdir(REPO.split("/")[1] + "/" + PROJECT)

## Step 1: install the pinned packages

This cell installs the exact versions from `requirements.txt`. Colab may ask you to restart the session. If it does, click **Restart session**, then run the cells again from the top (Step 0 onward).

In [ ]:
%pip install -q langgraph==0.6.7 langchain-core==0.3.72 langchain-google-genai==2.1.8 python-dotenv==1.1.1

## Step 2: API key (Colab Secrets)

Here is how to add your key, in plain words:

1. Click the **key icon** in the left sidebar (Secrets).
2. Click **Add new secret**. Name it exactly `GEMINI_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for that secret.

Never paste the key into a cell. Keyword routing runs without a key, so the main graph works with no key. Only the optional LLM routing fallback (Step 4) needs one. No key yet? Skip this step.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. Keyword routing still runs. Details:", e)

print("key found:", bool(os.environ.get("GEMINI_API_KEY", "").strip()))

### Step 1: Load the key
**What:** Read `GEMINI_API_KEY` from `.env`, the same way projects 04 and 07 do.
**Why:** Keys belong in a file you do not share. The key is optional, so the graph still runs without it.

In Colab the key comes from Colab Secrets (Step 2 above). This cell also looks for a `.env` file in the current folder, so the same code works here and on your computer.

In [ ]:
import logging
import os
import re
import sys
from pathlib import Path
from typing import TypedDict

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph

# Step 1: Load GEMINI_API_KEY from the .env file in this folder.
# Colab has no __file__, so this uses the current folder instead.
load_dotenv(Path.cwd() / ".env")

log = logging.getLogger("router_graph")

# Model names and free-tier limits change. Check Google AI Studio for the current name.
MODEL_NAME = "gemini-2.5-flash"

# Step 5: Any refund above this amount (in dollars) is blocked by the guardrail.
REFUND_LIMIT = 100.0

### Step 2: Define the typed state
**What:** A `TypedDict` named `RouterState` lists the four keys every node can see: `request`, `category`, `reply`, and `blocked`.
**Why:** A typed state is the shared memory of a graph. Each node reads the state and returns only the keys it changes. LangGraph merges those keys into the state for the next node.

In [ ]:
# Step 2: The typed state. Every node reads it and returns only the keys it changes.
class RouterState(TypedDict):
    request: str
    category: str
    reply: str
    blocked: bool

### Step 3: The router, with keyword rules first
**What:** `keyword_category` counts billing words and technical words in the request. The category with more matches wins. A tie returns `None`.
**Why:** Keyword rules are fast, free, and work offline. Use them before any model call.

In [ ]:
# Step 3: Keyword rules. The category with more matches wins.
BILLING_WORDS = ("bill", "billing", "invoice", "charge", "charged", "refund", "payment", "price", "subscription")
TECHNICAL_WORDS = ("error", "crash", "bug", "login", "log in", "install", "slow", "broken", "not working")


def keyword_category(text):
    """Return "billing", "technical", or None when the keywords do not decide."""
    lowered = text.lower()
    billing = sum(1 for word in BILLING_WORDS if re.search(rf"\b{re.escape(word)}\b", lowered))
    technical = sum(1 for word in TECHNICAL_WORDS if re.search(rf"\b{re.escape(word)}\b", lowered))
    if billing > technical:
        return "billing"
    if technical > billing:
        return "technical"
    return None


# Step 3 checkpoint. Expected output: billing technical None
print(keyword_category("I need a refund"), keyword_category("app error"), keyword_category("hello"))

### Step 4: The optional LLM fallback
**What:** If the keywords do not decide, and `GEMINI_API_KEY` is set, `llm_category` asks `gemini-2.5-flash` to answer with one word. If the call fails, the router logs a warning and uses the default, `technical`.
**Why:** A model can handle requests that use no keywords. The fallback is only used when needed, so most runs make no model call.

The notebook turns this on only when Step 2 found a real key (see `main()` in Step 7).

In [ ]:
# Step 4: The optional LLM fallback.
# Set in main() (Step 7): True only when a real key is present and --offline is not used.
USE_LLM = False


def llm_category(text):
    """Ask Gemini to classify the request. Returns "billing", "technical", or None."""
    llm = ChatGoogleGenerativeAI(
        model=MODEL_NAME,
        google_api_key=os.environ["GEMINI_API_KEY"],
        temperature=0,
    )
    prompt = "Classify this customer request as exactly one word: billing or technical.\n\nRequest: " + text
    answer = str(llm.invoke(prompt).content).strip().lower()
    if "billing" in answer:
        return "billing"
    if "technical" in answer:
        return "technical"
    return None


def router_node(state):
    """Step 3 and 4: keywords first, then the LLM if enabled, then a safe default."""
    category = keyword_category(state["request"])
    source = "keywords"
    if category is None and USE_LLM:
        try:
            category = llm_category(state["request"])
            source = "llm"
        except Exception as error:  # for example a network or quota error
            log.warning("LLM routing failed (%s), so the default is used", error)
    if category is None:
        category = "technical"
        source = "default"
    log.info("router chose %s via %s", category, source)
    return {"category": category}


def route_by_category(state):
    """Conditional edge: the name of the next node."""
    return state["category"]

### Step 5: Specialists and the guardrail
**What:** `billing_agent` and `technical_agent` return canned replies. `guardrail_node` looks for a sentence of the form "refund $X" in the reply. If X is over `REFUND_LIMIT` (100), it replaces the reply with a blocked message and sets `blocked` to `True`.
**Why:** A guardrail is a rule that checks an answer before it leaves the system. Here the rule is a simple pattern check. Real guardrails can use policy engines, classifiers, or a human review step.

In [ ]:
# Step 5: Any refund above REFUND_LIMIT is blocked by the guardrail.
REFUND_PATTERN = re.compile(r"refund \$(\d+(?:\.\d+)?)", re.IGNORECASE)
AMOUNT_PATTERN = re.compile(r"\$(\d+(?:\.\d+)?)")


def billing_agent(state):
    """Specialist for money questions. Canned reply, no real account is touched."""
    match = AMOUNT_PATTERN.search(state["request"])
    if match:
        reply = f"I can refund ${float(match.group(1)):.2f} to your original payment method."
    else:
        reply = "Please tell me the amount you were charged so I can check it."
    return {"reply": reply}


def technical_agent(state):
    """Specialist for app problems. Canned reply."""
    return {"reply": "Please clear your browser cache, then try logging in again. If the error continues, send us the error text."}


def guardrail_node(state):
    """Step 5: block any answer that promises a refund over REFUND_LIMIT."""
    match = REFUND_PATTERN.search(state["reply"])
    if match and float(match.group(1)) > REFUND_LIMIT:
        blocked_reply = (
            f"Blocked by guardrail: refunds over ${REFUND_LIMIT:.0f} need a human agent. "
            "A person will review this request."
        )
        return {"reply": blocked_reply, "blocked": True}
    return {"blocked": False}

### Step 6: Wire the graph
**What:** Add the four nodes, set `router` as the entry point, add a conditional edge from `router` to `billing` or `technical`, add edges from both specialists to `guardrail`, and end after the guardrail.
**Why:** The edges are the control flow. Reading them tells you every path a request can take.

The `traced()` helper below is the logging wrapper from the Observability section of the README. It logs node names and state keys, not values.

In [ ]:
def traced(name, node_fn):
    """Observability: log the node name and the state KEYS only (not values, so no customer text is logged)."""

    def wrapper(state):
        log.info("enter node=%s state_keys=%s", name, sorted(state.keys()))
        update = node_fn(state)
        log.info("exit node=%s updated_keys=%s", name, sorted(update.keys()))
        return update

    return wrapper


def build_graph():
    """Step 6: add the nodes and the edges, then compile."""
    graph = StateGraph(RouterState)
    graph.add_node("router", traced("router", router_node))
    graph.add_node("billing", traced("billing", billing_agent))
    graph.add_node("technical", traced("technical", technical_agent))
    graph.add_node("guardrail", traced("guardrail", guardrail_node))
    graph.set_entry_point("router")
    graph.add_conditional_edges("router", route_by_category, {"billing": "billing", "technical": "technical"})
    graph.add_edge("billing", "guardrail")
    graph.add_edge("technical", "guardrail")
    graph.add_edge("guardrail", END)
    return graph.compile()

### Step 7: Invoke it
**What:** `main()` builds the graph and runs two example requests with `app.invoke`.
**Why:** `invoke` takes an initial state and returns the final state. You read the answer from the result.

Run this cell. It is the checkpoint: each node writes an `enter` and an `exit` log line, and each request prints one `Category` line and one `Reply` line. Colab has no `--offline` flag. The notebook uses Gemini only if you added a key in Step 2, and only when the keywords do not decide.

In [ ]:
# Step 7: build the graph and run the two example requests.
def main():
    global USE_LLM
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

    offline = "--offline" in sys.argv
    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    USE_LLM = not offline and bool(api_key) and api_key != "your-key-here"
    if USE_LLM:
        print(f"Mode: keywords first, then Gemini ({MODEL_NAME}) if the keywords do not decide")
    else:
        print("Mode: OFFLINE keyword routing only (no network calls)")

    app = build_graph()
    examples = [
        "I was charged $250 twice, please refund the payment.",
        "The app shows an error when I try to log in.",
    ]
    for request in examples:
        result = app.invoke({"request": request, "category": "", "reply": "", "blocked": False})
        print(f"\nRequest: {request}")
        print(f"Category: {result['category']}  Blocked: {result['blocked']}")
        print(f"Reply: {result['reply']}")
    return 0


main()

## MCP (stretch, not in this notebook)

The MCP section needs a local terminal and the optional `requirements-mcp.txt`. It is not run in Colab, and this notebook does not install the MCP packages. To try it on your own computer, read the [STRETCH: MCP section of the README](README.md#stretch-mcp-model-context-protocol).

## Checkpoints and what you learned

- After Step 3: the check prints `billing technical None`
- After Step 5: a refund of $250 is blocked. A short billing question with no amount is not
- After Step 6: each request prints one `Category` line and one `Reply` line
- After Step 7: each node writes an `enter` and an `exit` log line

You learned how to describe a graph of nodes and edges with a typed state, route a request with keyword rules and an optional model fallback, add a guardrail node that can stop an answer, and log the path a request took.

To keep going, try the stretch goals in the project README.

Run locally instead? See the [README setup section](README.md#setup).